In [ ]:
# imports and loading data

from pathlib import Path
import os
import re
import json
import pandas as pd
import numpy as np
import random

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from scipy.stats import mannwhitneyu
import statsmodels.api as sm


# resolving project root

PROJECT_ROOT = Path.cwd().parent
print("Project root:", PROJECT_ROOT)

os.chdir(PROJECT_ROOT)


# setting seed for reproducibility

SEED = 42
random.seed(SEED)
np.random.seed(SEED)



In [ ]:
# load in samples 

SAMPLES_PATH = Path("data/generated/raw/samples.jsonl")

rows = []
with open(SAMPLES_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
df.shape, df.columns

In [ ]:
# parsing multi-label targets
# converting columns into list format 

df["label_list"] = df["labels"].str.split("+")
df["label_list"].head()


In [ ]:
# sanity checking unique labels

sorted({lab for labs in df["label_list"] for lab in labs})


In [5]:
# train/val/test splitting by job_id

job_ids = np.array(sorted(df["job_id"].unique()))
rng = np.random.default_rng(SEED)
rng.shuffle(job_ids)

n = len(job_ids)
train_ids = set(job_ids[: int(0.8 * n)])
val_ids   = set(job_ids[int(0.8 * n): int(0.9 * n)])
test_ids  = set(job_ids[int(0.9 * n):])

def assign_split(j):
    if j in train_ids: return "train"
    if j in val_ids:   return "val"
    return "test"

df["split"] = df["job_id"].map(assign_split)
df["split"].value_counts(), df["split"].value_counts(normalize=True)


# saving the splits

Path("splits").mkdir(exist_ok=True)
pd.Series(sorted(train_ids)).to_csv("splits/train_job_ids.csv", index=False)
pd.Series(sorted(val_ids)).to_csv("splits/val_job_ids.csv", index=False)
pd.Series(sorted(test_ids)).to_csv("splits/test_job_ids.csv", index=False)


## note to self: run this instead if there are already savd files
#train_ids = set(pd.read_csv("splits/train_job_ids.csv")["job_id"])
#val_ids   = set(pd.read_csv("splits/val_job_ids.csv")["job_id"])
#test_ids  = set(pd.read_csv("splits/test_job_ids.csv")["job_id"])


In [ ]:
# sanity check - works!:)

df.groupby("split")["label_list"].apply(lambda s: pd.Series([x for xs in s for x in xs]).value_counts())

In [ ]:
# vectorising text and binarising labels 

X_text = df["text"].values
y_lists = df["label_list"].values

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(y_lists)

mlb.classes_, Y.shape


In [ ]:
# training baseline TF-IDF + OvR logistic regression

# split indices
train_mask = df["split"].eq("train").values
val_mask   = df["split"].eq("val").values
test_mask  = df["split"].eq("test").values

tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
)

X_train = tfidf.fit_transform(X_text[train_mask])
X_val   = tfidf.transform(X_text[val_mask])
X_test  = tfidf.transform(X_text[test_mask])

clf = OneVsRestClassifier(
    LogisticRegression(
        max_iter=2000,
        solver="liblinear",
        random_state=SEED,
    )
)

clf.fit(X_train, Y[train_mask])


In [ ]:
# eval (overall and per-label eval)


Y_val_pred  = clf.predict(X_val)
Y_test_pred = clf.predict(X_test)

print("VAL micro F1:", f1_score(Y[val_mask], Y_val_pred, average="micro"))
print("VAL macro F1:", f1_score(Y[val_mask], Y_val_pred, average="macro"))

print("\nTEST micro F1:", f1_score(Y[test_mask], Y_test_pred, average="micro"))
print("TEST macro F1:", f1_score(Y[test_mask], Y_test_pred, average="macro"))

print("\nPer-label report (TEST):")
print(classification_report(Y[test_mask], Y_test_pred, target_names=mlb.classes_))


In [ ]:
leak_terms = ["nociception", "temperature", "pressure", "vibration", "dataset", "label"]
for t in leak_terms:
    n = df["text"].str.lower().str.contains(t).sum()
    print(t, n)


In [ ]:
# testing whether the appearance of the labels drives  F1 scores


def mask_label_words(text):
    t = text
    # masking common label-ish tokens
    t = re.sub(r"\bpressure\b", "___", t, flags=re.IGNORECASE)
    t = re.sub(r"\bvibration(s)?\b", "___", t, flags=re.IGNORECASE)
    t = re.sub(r"\btemperature\b", "___", t, flags=re.IGNORECASE)
    return t

# building masked X_test
masked_test_text = np.array([mask_label_words(t) for t in df.loc[test_mask, "text"].values])
X_test_masked = tfidf.transform(masked_test_text)

Y_test_pred_masked = clf.predict(X_test_masked)

print("Original TEST micro F1:", f1_score(Y[test_mask], Y_test_pred, average="micro"))
print("Masked   TEST micro F1:", f1_score(Y[test_mask], Y_test_pred_masked, average="micro"))


In [ ]:
# performance breakdown by experimental axes 


test_df = df[df["split"] == "test"].copy()

# predictions for test rows aligned with test_df order
test_df["pred_list"] = list(mlb.inverse_transform(Y_test_pred))
test_df["true_list"] = list(mlb.inverse_transform(Y[test_mask]))

def f1_for_subset(mask):
    idx = test_df.index[mask]
    if len(idx) == 0:
        return np.nan
    # re-vectorize subset into matrices aligned to predictions
    sub_true = mlb.transform(test_df.loc[idx, "true_list"])
    sub_pred = mlb.transform(test_df.loc[idx, "pred_list"])
    return f1_score(sub_true, sub_pred, average="micro")

# Example: by anchor_regime
for regime in ["strict", "paraphrase", "drift"]:
    m = test_df["anchor_regime"].eq(regime)
    print(regime, "micro F1:", f1_for_subset(m))

# Example: literal vs metaphorical
for lit in [0, 1]:
    m = test_df["literal"].eq(lit)
    print("literal" if lit==1 else "metaphor", "micro F1:", f1_for_subset(m))


### FULL EVAL BELOW

In [13]:
# one unified eval table across all dimensions


test_mask = df["split"].eq("test").values
test_df = df.loc[test_mask].copy().reset_index(drop=True)

Y_test_true = Y[test_mask]
Y_test_pred = clf.predict(X_test)

test_df["true_list"] = list(mlb.inverse_transform(Y_test_true))
test_df["pred_list"] = list(mlb.inverse_transform(Y_test_pred))

test_df["k_mods"] = test_df["labels"].astype(str).str.split("+").apply(len)

def micro_f1_subset(mask):
    mask = np.array(mask)
    if mask.sum() == 0:
        return np.nan
    sub_true = mlb.transform(test_df.loc[mask, "true_list"])
    sub_pred = mlb.transform(test_df.loc[mask, "pred_list"])
    return f1_score(sub_true, sub_pred, average="micro", zero_division=0)

def summarise_by(col, values=None):
    out = []
    if values is None:
        values = sorted(test_df[col].dropna().unique().tolist())
    for v in values:
        m = test_df[col].eq(v)
        out.append({"axis": col, "value": v, "n": int(m.sum()), "micro_f1": micro_f1_subset(m)})
    return pd.DataFrame(out)

all_tables = []


In [ ]:
# single-axis breakdowns
all_tables.append(summarise_by("anchor_regime", ["strict","paraphrase","drift"]))
all_tables.append(summarise_by("literal", [0,1]))
all_tables.append(summarise_by("specificity", [0,1]))
all_tables.append(summarise_by("consistent", [0,1]))
all_tables.append(summarise_by("k_mods", [1,2,3]))

axis_table = pd.concat(all_tables, ignore_index=True)
axis_table



In [ ]:
# interaction breakdown helper

def interaction_f1(name, mask):
    mask = np.array(mask)
    return {"group": name, "n": int(mask.sum()), "micro_f1": micro_f1_subset(mask)}

rows = []

# 2-way interactions
for r in ["strict","paraphrase","drift"]:
    for lit in [0,1]:
        rows.append(interaction_f1(f"{r} & literal={lit}", (test_df["anchor_regime"].eq(r) & test_df["literal"].eq(lit))))

for r in ["strict","paraphrase","drift"]:
    for k in [1,2,3]:
        rows.append(interaction_f1(f"{r} & k_mods={k}", (test_df["anchor_regime"].eq(r) & test_df["k_mods"].eq(k))))

for lit in [0,1]:
    for k in [1,2,3]:
        rows.append(interaction_f1(f"literal={lit} & k_mods={k}", (test_df["literal"].eq(lit) & test_df["k_mods"].eq(k))))

# 3-way "hard mode"
rows.append(interaction_f1("drift & metaphor & k_mods=3",
                           (test_df["anchor_regime"].eq("drift") & test_df["literal"].eq(0) & test_df["k_mods"].eq(3))))

rows.append(interaction_f1("paraphrase & metaphor & k_mods=3",
                           (test_df["anchor_regime"].eq("paraphrase") & test_df["literal"].eq(0) & test_df["k_mods"].eq(3))))

rows.append(interaction_f1("strict & inconsistent",
                           (test_df["anchor_regime"].eq("strict") & test_df["consistent"].eq(0))))

inter_table = pd.DataFrame(rows).sort_values(["micro_f1", "n"], ascending=[True, False])
inter_table.head(25)


In [ ]:
# multi-label confusion diagnostics

for i, lab in enumerate(mlb.classes_):
    cm = confusion_matrix(Y_test_true[:, i], Y_test_pred[:, i])
    tn, fp, fn, tp = cm.ravel()
    print(f"\n{lab}")
    print("TN FP\nFN TP")
    print(cm)
    print("FP rate:", fp / (fp + tn + 1e-9), "FN rate:", fn / (fn + tp + 1e-9))


In [ ]:
# hard subset evaluation, aka excluding explicit tokens

no_explicit = ~test_df["text"].str.lower().str.contains(r"\bpressure\b|\bvibration(s)?\b|\btemperature\b")
print("Fraction of test with NO explicit tokens:", no_explicit.mean())
print("Micro F1 on no-explicit subset:", micro_f1_subset(no_explicit))


In [ ]:
# qualitative failure analysis - inspecting the hardest examples

errors = []
for i in range(len(test_df)):
    t = set(test_df.loc[i, "true_list"])
    p = set(test_df.loc[i, "pred_list"])
    if t != p:
        errors.append(i)

len(errors), len(test_df)


In [ ]:
# sampling 20 error cases with metadata

sample_idx = np.random.default_rng(SEED).choice(errors, size=min(20, len(errors)), replace=False)

cols = ["text","labels","anchor_regime","literal","specificity","consistent","k_mods","true_list","pred_list"]
test_df.loc[sample_idx, cols]


In [ ]:
# example: temp missed 

temp_missed = []
temp_i = list(mlb.classes_).index("temperature")
for i in range(len(test_df)):
    if Y_test_true[i, temp_i] == 1 and Y_test_pred[i, temp_i] == 0:
        temp_missed.append(i)

test_df.loc[temp_missed[:20], cols]


## Second part of the evaluation: Cross-modal salience and its' influence on classification

In [ ]:
# loading strict only test set and model correctness

strict_test = test_df[test_df["anchor_regime"].eq("strict")].copy()

# exact-match correctness (meaning: all labels correct for that sample)
strict_test["exact_match"] = strict_test.apply(
    lambda r: set(r["true_list"]) == set(r["pred_list"]), axis=1
)

strict_test[["anchors_planned","labels","exact_match"]].head()


In [22]:
# loading lancaster sensorimotor norms and prepping lookup

lex_path = PROJECT_ROOT / "data" / "raw" / "Sensorimotor_norms_24Jan2026.csv"
lex = pd.read_csv(lex_path)

# normalising Word field for matching
lex["Word_norm"] = lex["Word"].astype(str).str.strip().str.upper()

# columns we need
cols = [
    "Word_norm",
    "Haptic.mean", "Visual.mean", "Auditory.mean", "Olfactory.mean", "Gustatory.mean", "Interoceptive.mean"
]
lex_small = lex[cols].copy()

# make a lookup dict row-wise
lex_lookup = lex_small.set_index("Word_norm").to_dict(orient="index")


In [23]:
# parsing anchors_planned and matching to lexicon

def parse_anchors_planned(s):
    if pd.isna(s) or str(s).strip() == "":
        return []
    return [a.strip().upper() for a in str(s).split(";") if a.strip()]

other_modal_cols = ["Visual.mean","Auditory.mean","Olfactory.mean","Gustatory.mean","Interoceptive.mean"]

def anchor_to_scores(anchor):
    a = anchor.strip().upper()
    if a in lex_lookup:
        d = lex_lookup[a]
        h = d["Haptic.mean"]
        other_max = max(d[c] for c in other_modal_cols)
        return {"anchor": a, "found": True, "haptic": h, "other_max": other_max, "dominance": h - other_max}

    # fallback for multiword anchors: split and try parts
    parts = a.replace("-", " ").split()
    part_rows = [lex_lookup[p] for p in parts if p in lex_lookup]

    if len(part_rows) == 0:
        return {"anchor": a, "found": False, "haptic": np.nan, "other_max": np.nan, "dominance": np.nan}

    # picking the part with highest haptic mean 
    best = max(part_rows, key=lambda r: r["Haptic.mean"])
    h = best["Haptic.mean"]
    other_max = max(best[c] for c in other_modal_cols)
    return {"anchor": a, "found": True, "haptic": h, "other_max": other_max, "dominance": h - other_max}

# expanding anchors per row
strict_test["anchors_list"] = strict_test["anchors_planned"].apply(parse_anchors_planned)


In [ ]:
# computing dominance aggregates per sample

def row_dominance_stats(anchor_list):
    rows = [anchor_to_scores(a) for a in anchor_list]
    doms = [r["dominance"] for r in rows if np.isfinite(r["dominance"])]
    found = sum(1 for r in rows if r["found"])
    total = len(anchor_list)

    if len(doms) == 0:
        return pd.Series({
            "n_anchors": total,
            "n_found": found,
            "dominance_mean": np.nan,
            "dominance_min": np.nan
        })

    return pd.Series({
        "n_anchors": total,
        "n_found": found,
        "dominance_mean": float(np.mean(doms)),
        "dominance_min": float(np.min(doms))
    })

strict_test[["n_anchors","n_found","dominance_mean","dominance_min"]] = strict_test["anchors_list"].apply(row_dominance_stats)
strict_test[["n_anchors","n_found","dominance_mean","dominance_min","exact_match"]].head()


In [ ]:
# sanity check

coverage = strict_test["n_found"].sum() / strict_test["n_anchors"].sum()
coverage


In [ ]:
## Actual analysis

strict_ok = strict_test.loc[strict_test["exact_match"] == True, "dominance_mean"].dropna()
strict_bad = strict_test.loc[strict_test["exact_match"] == False, "dominance_mean"].dropna()

summary = pd.DataFrame({
    "Group": ["Correctly classified", "Incorrectly classified"],
    "Mean dominance": [strict_ok.mean(), strict_bad.mean()],
    "Median dominance": [strict_ok.median(), strict_bad.median()],
    "N": [len(strict_ok), len(strict_bad)]
})

summary


In [ ]:
# nonparam test

u_stat, p_val = mannwhitneyu(
    strict_ok,
    strict_bad,
    alternative="greater"
)

print("Mann–Whitney U test (greater dominance → correct classification)")
print(f"U statistic: {u_stat:.1f}")
print(f"p-value:     {p_val:.4f}")
print(f"N correct:   {len(strict_ok)}")
print(f"N incorrect: {len(strict_bad)}")



In [ ]:
# effect size

effect_size = u_stat / (len(strict_ok) * len(strict_bad))
print(f"Rank-biserial effect size (U / (n1·n2)): {effect_size:.3f}")


In [ ]:
# logreg

tmp = strict_test.dropna(subset=["dominance_mean"]).copy()
X = sm.add_constant(tmp["dominance_mean"])
y = tmp["exact_match"].astype(int)

m = sm.Logit(y, X).fit(disp=False)
m.summary()


In [30]:
# for plotting later
from pathlib import Path
Path("data/processed").mkdir(parents=True, exist_ok=True)

axis_table.to_csv("data/processed/axis_table_baseline.csv", index=False)
strict_test.to_csv("data/processed/strict_test_baseline.csv", index=False)
print("Saved baseline tables.")


Saved baseline tables.


In [31]:
# threshold sweep plot get-ready (find the plotting code in the transformer model notebook)

def microf1_threshold_sweep(probs: np.ndarray, Y_true: np.ndarray, thresholds):
    probs = np.asarray(probs)
    Y_true = np.asarray(Y_true).astype(int)
    f1s = []
    for t in thresholds:
        Y_pred = (probs >= t).astype(int)
        f1s.append(f1_score(Y_true, Y_pred, average="micro", zero_division=0))
    return np.asarray(f1s)

# 1) probabilities
probs_base = clf.predict_proba(X_test)  # shape (N,4)

# 2) sweep
thresholds = np.linspace(0.05, 0.95, 19)
f1_base = microf1_threshold_sweep(probs_base, Y_test_true, thresholds)

best_t = thresholds[np.argmax(f1_base)]
best_f1 = f1_base.max()
print(f"Baseline best micro-F1 = {best_f1:.4f} at t = {best_t:.2f}")

# 3) save
out_dir = Path("results/threshold_sweeps")
out_dir.mkdir(parents=True, exist_ok=True)

np.savez(
    out_dir / "threshold_sweep_baseline.npz",
    thresholds=thresholds,
    f1=f1_base,
    probs=probs_base,        
    Y_true=Y_test_true      
)

print("Saved:", out_dir / "threshold_sweep_baseline.npz")


Baseline best micro-F1 = 0.9186 at t = 0.40
Saved: results/threshold_sweeps/threshold_sweep_baseline.npz
